In [0]:
df = spark.sql("select * from gizmobox.bronze.py_customers")
display(df)


In [0]:
df = spark.table("gizmobox.bronze.py_customers")
display(df)

In [0]:
df =( spark.table("gizmobox.bronze.py_customers")
    .filter("customer_id is not null")
)
display(df)

In [0]:
df = spark.table("gizmobox.bronze.py_customers")
df_filtered = df.filter(df.customer_id.isNotNull())
display(df_filtered)

In [0]:
df_filtered = df.where(df.customer_id.isNotNull())
display(df_filtered)

In [0]:
df_distinct = df_filtered.drop_duplicates()
display(df_distinct)

In [0]:
from pyspark.sql import functions as F  
df_max_ts = df_distinct.groupBy("customer_id") \
                       .agg(F.max("created_timestamp").alias('max_created_timestamp'))

display(df_max_ts)

In [0]:
df_distinct_customer = (
                        df_distinct
                        .join(df_max_ts, (df_distinct.customer_id == df_max_ts.customer_id) & 
                                         (df_distinct.created_timestamp == df_max_ts.max_created_timestamp), 'inner')
                        .select(df_distinct['*'])
)
display(df_distinct_customer)

In [0]:
df_casted_customers = (
                        df_distinct_customer
                            .select(df_distinct_customer.created_timestamp.cast('timestamp'),
                                    df_distinct_customer.customer_id,
                                    df_distinct_customer.customer_name,
                                    df_distinct_customer.date_of_birth.cast('date'),
                                    df_distinct_customer.email,
                                    df_distinct_customer.member_since.cast('date'),
                                    df_distinct_customer.telephone)
)
display(df_casted_customers)

In [0]:
df_casted_customers.writeTo("gizmobox.silver.py_customers").createOrReplace()

In [0]:
%sql
select * from gizmobox.silver.py_customers